# EXERCÍCIO 03

**Vinícius Sousa Dutra · Número USP 13686257**  

## Introdução

O objetivo é comparar o desempenho de uma rede MLP contra uma RBF na classificação da base Wine usada no projeto 01,separando os dados em treinamento (80%) e teste (20%). Iremos usar a biblioteca `scikit-learn` tanto para coletar os dados quanto para treinar as redes, os imports e constantes globais se encontram no bloco abaixo

In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

SEED = 123456789

## Dados e partições
Iremos normalizar os dados para que cada atributo tenha $\mu=0$ e $\sigma^2=1$ no treino

In [6]:
atributos, classes = load_wine(return_X_y=True)

X = np.asarray(atributos, dtype=np.float64)
y = np.asarray(classes, dtype=np.int64)

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

normalizador = StandardScaler()
X_treino = np.asarray(
    normalizador.fit_transform(X_treino),
    dtype=np.float64,
)
X_teste = np.asarray(normalizador.transform(X_teste), dtype=np.float64)

print(f"Amostras: {len(y)} \t Treinamento: {len(y_treino)} \t Teste: {len(y_teste)} ")

Amostras: 178 	 Treinamento: 142 	 Teste: 36 


## Rede MLP

A MLP possui arquitetura 13–13–3: treze entradas, uma única camada intermediária com treze neurônios e três saídas. A largura da camada intermediária foi igualada à dimensão da entrada por simplicidade, não foi feita busca de hiperparâmetros ou algo do tipo.

In [8]:
mlp = MLPClassifier(
    hidden_layer_sizes=(X_treino.shape[1],),
    activation="logistic",
    solver="sgd",
    max_iter=10_000,
    random_state=SEED,
).fit(X_treino, y_treino)
acurácia_mlp = mlp.score(X_teste, y_teste)

## Rede RBF
Como o exercício já oferece uma hipotése simplificadora para a parte de escolher os centros e as larguras, iremos usar um neurônio da rede RBF para cada classe. Para a classe $k$, o centro $c_k$ é a média dos exemplos de treinamento daquela classe e a largura $\sigma_k$ é a raiz da média das distâncias quadráticas até o centro:

$$c_k = \frac{1}{n_k} \sum_{i:y_i=k} x_i, \qquad \sigma_k = \sqrt{\frac{1}{n_k} \sum_{i:y_i=k} \lVert x_i-c_k \rVert^2}.$$

Escolhemos a ativação gaussiana para cada neurônio:

$$\phi_k(x) = \exp\left(-\frac{\lVert x-c_k \rVert^2}{2\sigma_k^2}\right).$$

As três ativações formam a entrada de uma camada de saída multiclasse, ajustada pelo `LogisticRegression`.

In [17]:
from numpy.typing import NDArray
type MatrizAtributos = NDArray[np.float64]
type VetorClasses = NDArray[np.int64]
type VetorReais = NDArray[np.float64]


def estimar_centros_e_larguras(
    X: MatrizAtributos,
    y: VetorClasses,
) -> tuple[MatrizAtributos, VetorReais]:
    grupos = tuple(X[y == classe] for classe in np.unique(y))
    ## os centros são a média dos dados para cada grupo
    centros = np.stack(tuple(grupo.mean(axis=0) for grupo in grupos))
    ## as larguras são os desvios padrões dos vetores de cada grupo 
    ## em relação ao centro
    larguras = np.array(
        [
            np.sqrt(np.mean(np.sum((grupo - centro) ** 2, axis=1)))
            for grupo, centro in zip(grupos, centros, strict=True)
        ],
        dtype=np.float64,
    )
    return np.asarray(centros, dtype=np.float64), larguras


def rbf(
    X: MatrizAtributos,
    centros: MatrizAtributos,
    larguras: VetorReais,
) -> MatrizAtributos:
    diferenças = X[:, np.newaxis, :] - centros[np.newaxis, :, :]
    distâncias_quadradas = np.sum(diferenças**2, axis=2)
    return np.exp(-distâncias_quadradas / (2.0 * larguras**2))


centros, larguras = estimar_centros_e_larguras(X_treino, y_treino)
X_treino_rbf = rbf(X_treino, centros, larguras)
X_teste_rbf = rbf(X_teste, centros, larguras)


saída_rbf = LogisticRegression().fit(X_treino_rbf,y_treino)

acurácia_rbf = saída_rbf.score(X_teste_rbf, y_teste)

## Resultados

As acurácias obtidas pelas duas redes sobre os 36 exemplos de teste são:

In [18]:
print(
    "Acurácia no conjunto de teste:\n"
    f"  MLP: {acurácia_mlp:.2%}\n"
    f"  RBF: {acurácia_rbf:.2%}"
)

Acurácia no conjunto de teste:
  MLP: 100.00%
  RBF: 97.22%


## Conclusão

Como só existem 36 dados de teste, a diferença de só 3% entre o MLP e o RBF se deve a o erro de classificação do RBF em somente um exemplo, isso não é o suficiente para concluir no geral se uma rede RBF é menos eficiente do que uma MLP, para esse caso específico pode se dizer que ambas tem acurácia semelhante.